# Day 10 - Async Python (doing many slow things at once)
Vidya Sankalp | Applied GenAI Engineering

> **This day builds directly on Day 08 and Day 09.** On Day 08 you queried a **Postgres database**. On Day 09 you called **real APIs** over HTTP (DummyJSON and OpenAI). Both of those are *slow* - they make your program wait. Today you learn to do many of those slow things **at the same time**.

> **No pretend waiting.** The slow work here is the real database queries and real HTTP calls you already know - not a fake `sleep()`. That is the whole reason we taught database and HTTP first.

> **What you'll learn:** why waiting is slow -> `async`/`await` -> `httpx.AsyncClient` -> `asyncio.gather` (many at once) -> handling failures -> background work -> talking to Postgres the async way (`asyncpg`) and the 'use what you have' way (`to_thread` + your Day 08 code) -> streaming an AI reply word by word.

> **You need internet** for the HTTP/OpenAI cells (they call real servers), and the Postgres cells need a running database (same `.env` as Day 08). Each part says what it needs, and skips cleanly if it's not there.

> **Jupyter note:** a notebook already runs an event loop, so we use **top-level `await`** in cells. In a real `.py` file you'd wrap the start in `asyncio.run(main())` (shown at the end).


---


## 0 - One-time setup

We need `httpx` (Day 09) and, for the async database part, `asyncpg`. Install once if needed.


In [ ]:
# Run once if these libraries aren't installed yet. Remove the # to run it.
#   httpx   = make HTTP calls (from Day 09)
#   asyncpg = talk to Postgres the async way (used in section 5)
# !pip install -q httpx asyncpg python-dotenv

import asyncio   # asyncio = Python's built-in toolkit for doing many waiting-jobs at once
import httpx     # httpx    = the HTTP tool from Day 09 (it also has an async version)
import time      # time     = a stopwatch, to measure how long things take
import os        # os       = used later to read database settings from your environment

BASE = 'https://dummyjson.com'      # the same real API server we used on Day 09
print('Ready.')


---


## 1 - The Problem: Slow Things, One After Another

Remember the end of Day 09? Five slow calls, one after another, took about 2.5 seconds. Let's feel that pain again with **real HTTP calls** - we ask DummyJSON for 5 products, and we add `?delay=300` so each one takes about a third of a second (like a real, slightly slow server).


In [ ]:
# The SLOW way: fetch 5 products one after another (ordinary, NON-async httpx from Day 09).
start = time.perf_counter()              # perf_counter() = a precise stopwatch reading, in seconds
titles = []
for pid in [1, 2, 3, 4, 5]:              # do each id in turn
    # params={'delay': 300}  ->  ask the server to wait 300 milliseconds before replying,
    #                            so each call is realistically 'slow' (about a third of a second).
    # timeout=15.0           ->  wait at most 15 seconds for each reply before giving up.
    r = httpx.get(f'{BASE}/products/{pid}', params={'delay': 300}, timeout=15.0)
    titles.append(r.json()['title'])     # pull the title out of each reply
print('Got:', titles)
# Because each call must FINISH before the next one STARTS, the delays add up:
print(f'One after another took {time.perf_counter()-start:.1f}s  (~5 x 0.3s)')


Every call had to *finish* before the next one *started*. But while we wait for DummyJSON to reply, our program is doing nothing useful. Async lets all 5 waits happen together.


---


## 2 - The Two New Words: `async` and `await`

| Word | Plain meaning |
|------|---------------|
| `async def` | marks a function as the 'smart waiting' kind - it can pause and let other work happen |
| `await` | 'I have to wait here - go do something else useful, come back when this is ready' |

For HTTP, the async version of the Day 09 client is **`httpx.AsyncClient`**. Same calls (`.get`, `.post`), but you `await` them, and you open it with `async with` so it cleans up after itself.


In [ ]:
# The async version of the Day 09 client is httpx.AsyncClient. Three differences to notice:
#   1) the function is defined with 'async def' (it's the 'smart waiting' kind)
#   2) we open the client with 'async with' (it cleans itself up when the block ends)
#   3) we put 'await' before the call (pause here, let other work run while we wait)
async def get_one_title(pid):
    #   base_url=BASE  ->  set the home address once, so we can pass just '/products/1' to .get()
    #   timeout=15.0   ->  wait at most 15 seconds for a reply (same meaning as Day 09)
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        r = await client.get(f'/products/{pid}')   # 'await' = run this and wait for the reply
        return r.json()['title']

# In a notebook we are allowed to use 'await' directly in a cell (Jupyter runs the event loop).
title = await get_one_title(1)
print('Product 1 is:', title)


> Opening a fresh `AsyncClient` for every call is wasteful (like dialing a new phone line each time). In the next section we open **one** client and reuse it for all the calls - which is also what makes them run together.


---


## 3 - `asyncio.gather`: Many Calls at Once

`asyncio.gather` takes several jobs, runs them **all at the same time**, and gives back all the answers together (in order). This is the fix for section 1's slow loop.


In [ ]:
# The FAST way: open ONE client and fire all 5 requests together with asyncio.gather.
async def get_titles(pids):
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        # an inner helper that fetches ONE product's title
        async def one(pid):
            r = await client.get(f'/products/{pid}', params={'delay': 300})  # delay = pretend-slow
            return r.json()['title']
        # Build a list of jobs - one 'one(pid)' per id - WITHOUT awaiting them yet.
        jobs = [one(pid) for pid in pids]
        # asyncio.gather(*jobs) runs them ALL at the same time and waits for every one to finish.
        #   the *  unpacks the list  [one(1), one(2), ...]  into separate arguments for gather.
        #   the results come back as a list, in the SAME ORDER as the ids we passed in.
        return await asyncio.gather(*jobs)

start = time.perf_counter()
titles = await get_titles([1, 2, 3, 4, 5])
print('Got:', titles)
# All 5 waits overlapped, so the total is about ONE delay, not five added together:
print(f'All at once took {time.perf_counter()-start:.1f}s  (NOT ~1.5s!)')


Five slow calls finished in about the time of **one**, because the waits overlapped. Same real API, same data, just done concurrently. *This* is why async matters - and you can feel it because the waiting is real.


---


## 4 - When One Call Fails: `return_exceptions=True`

Remember 404s from Day 09 (asking for a product that doesn't exist)? If one job in `gather` raises an error, it normally ruins the whole batch. Adding `return_exceptions=True` keeps the good results and hands you the errors as values instead.


In [ ]:
async def get_or_error(pids):
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        async def one(pid):
            r = await client.get(f'/products/{pid}')
            r.raise_for_status()   # turn a 404 (not found) into an error we can catch (from Day 09)
            return r.json()['title']
        # return_exceptions=True changes how gather handles a job that fails:
        #   - WITHOUT it: if any job raises, the whole gather stops and raises immediately.
        #   - WITH it:    a failed job's error is returned as a VALUE in the results list,
        #                 and all the good results still come back. Nothing is lost.
        return await asyncio.gather(
            *[one(pid) for pid in pids],
            return_exceptions=True,
        )

# id 99999 does not exist -> that one becomes an error; the others are fine.
results = await get_or_error([1, 2, 99999, 4])
for pid, res in zip([1, 2, 99999, 4], results):   # zip pairs each id with its result
    # isinstance(res, Exception) is True when this slot holds an error instead of data.
    if isinstance(res, Exception):
        print(f'  product {pid}: FAILED ({type(res).__name__})')
    else:
        print(f'  product {pid}: {res}')


---


## 5 - Talking to Postgres the Async Way

On Day 08 you used **`psycopg2`** to query Postgres. `psycopg2` is *blocking* - it freezes while waiting, which is bad in async code. There are two good ways to handle databases in async:

| Way | What it is | When to use |
|-----|------------|-------------|
| **`asyncpg`** | a database library built *for* async Postgres | best choice for new async code |
| **`to_thread` + your Day 08 `psycopg2` code** | run the old blocking code on a side worker | when you want to reuse code you already have |

We'll show both. They need a running Postgres (the same `.env` as Day 08). If you don't have one connected, these cells just print a skip message.


### 5a - The proper async way: `asyncpg`

`asyncpg` mirrors what you already know: make a **pool** (a set of reusable connections), `acquire()` one with `async with`, and `await` your query. Postgres uses `$1, $2` for placeholders (instead of psycopg2's `%s`).


In [ ]:
import os
# Try to import asyncpg and load your database settings from a .env file (same as Day 08).
try:
    import asyncpg                       # the async Postgres library
    from dotenv import load_dotenv       # reads a .env file into os.environ
    load_dotenv()                        # actually load it
    # all(...) is True only if BOTH DB_NAME and DB_USER are set (a quick 'is a DB configured?' check)
    HAVE_PG = all(os.environ.get(k) for k in ['DB_NAME', 'DB_USER'])
except Exception:
    HAVE_PG = False                      # if asyncpg isn't installed or .env is missing, skip DB cells
print('Postgres available:', HAVE_PG)


In [ ]:
# We keep ONE pool of connections and reuse it. A 'pool' is a small set of ready-to-use database
# connections - opening a connection is slow, so we open a few once and share them.
_pg_pool = None                          # starts empty; we build it the first time we need it

async def get_pool():
    global _pg_pool                      # 'global' lets us update the module-level variable above
    if _pg_pool is None:                 # only build it once
        _pg_pool = await asyncpg.create_pool(   # create_pool opens the connections (so we await it)
            host     = os.environ.get('DB_HOST', 'localhost'),   # 2nd arg = default if not set
            port     = int(os.environ.get('DB_PORT', '5432')),   # int(...) because a port is a number
            database = os.environ.get('DB_NAME'),
            user     = os.environ.get('DB_USER'),
            password = os.environ.get('DB_PASSWORD'),
            min_size = 1,                # keep at least 1 connection ready
            max_size = 5,                # never open more than 5 at once
        )
    return _pg_pool

async def fetch_customer(customer_id):
    """Async version of Day 08's get_customer_by_id()."""
    pool = await get_pool()
    # 'async with pool.acquire()' BORROWS one connection and gives it back automatically at the end.
    async with pool.acquire() as conn:
        # await conn.fetchrow(SQL, value) runs the query and returns the FIRST matching row.
        #   $1  is the placeholder for the first value (Postgres style).
        #   Day 08's psycopg2 used %s; asyncpg uses $1, $2, $3, ...  Same idea: never glue values
        #   into SQL by hand (that risks SQL injection) - pass them separately and let the library do it.
        row = await conn.fetchrow(
            'SELECT * FROM customers WHERE customer_id = $1', customer_id)
    # asyncpg returns a 'Record' (a row). dict(row) turns it into a normal Python dictionary.
    return dict(row) if row else None    # None if no customer had that id

if HAVE_PG:
    one = await fetch_customer(1001)
    print('Fetched (async):', one)
else:
    print('(skipped - no Postgres connected)')


Now the payoff: fetch **several** customers **at once** with `gather`, exactly like we did for HTTP. Each lookup is a real database round-trip, and they overlap.


In [ ]:
if HAVE_PG:
    start = time.perf_counter()
    # Fetch three customers AT ONCE. Each fetch is a real database round-trip; gather overlaps them,
    # exactly like it overlapped the HTTP calls earlier.
    customers = await asyncio.gather(
        fetch_customer(1001),
        fetch_customer(1002),
        fetch_customer(1003),
    )
    for c in customers:
        print('  ', c)
    print(f'Three real DB lookups together took {time.perf_counter()-start:.2f}s')
else:
    print('(skipped - no Postgres connected)')


### 5b - The 'use what you already have' way: `to_thread`

Maybe you already wrote blocking `psycopg2` functions on Day 08 and don't want to rewrite them. `asyncio.to_thread` runs that blocking code on a side worker so it doesn't freeze the event loop. You keep your Day 08 function exactly as-is and just `await asyncio.to_thread(it)`.


In [ ]:
import psycopg2, psycopg2.extras

def get_customer_by_id_blocking(customer_id):
    """This is your Day 08 function, UNCHANGED - plain, blocking psycopg2."""
    conn = psycopg2.connect(             # psycopg2.connect = open a (blocking) DB connection
        host=os.environ.get('DB_HOST', 'localhost'),
        port=int(os.environ.get('DB_PORT', '5432')),
        dbname=os.environ.get('DB_NAME'),
        user=os.environ.get('DB_USER'),
        password=os.environ.get('DB_PASSWORD'),
    )
    # RealDictCursor makes each row come back as a dict instead of a plain tuple (Day 08).
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute('SELECT * FROM customers WHERE customer_id = %s', (customer_id,))  # %s = placeholder
        row = cur.fetchone()             # fetchone() = the first matching row (or None)
    conn.close()
    return dict(row) if row else None

async def fetch_customer_via_thread(customer_id):
    # The function above is BLOCKING - if we called it directly in async code it would freeze
    # everything. asyncio.to_thread runs it on a separate worker thread instead, so the event
    # loop stays free. We 'await' the worker's result.
    #   to_thread(FUNCTION, ARG)  ->  note we pass the function and its argument SEPARATELY,
    #                                 we do NOT call it ourselves (no parentheses on the function).
    return await asyncio.to_thread(get_customer_by_id_blocking, customer_id)

if HAVE_PG:
    # Even though the underlying code is blocking, gather still overlaps the waits:
    rows = await asyncio.gather(
        fetch_customer_via_thread(1001),
        fetch_customer_via_thread(1002),
    )
    for r in rows: print('  ', r)
else:
    print('(skipped - no Postgres connected)')


> **Which should you use?** For brand-new async code, `asyncpg` is cleaner and faster. For reusing existing blocking libraries (psycopg2, or any old sync code), `to_thread` is the bridge. Both are correct - now you know both.


---


## 6 - Mixing It All: Fetch (HTTP) + Validate (Pydantic from Day 08)

Let's combine all three days: fetch products concurrently over HTTP (Day 09 + today), then validate each one with a **Pydantic model** (Day 08). Clean data, fetched fast.


In [ ]:
from pydantic import BaseModel, Field

# A Pydantic model just like Day 08's ProductRecord. It describes the shape we EXPECT,
# and validates/cleans the data the API gives us.
class ProductRecord(BaseModel):
    id: int                              # must be a whole number
    title: str                           # must be text
    category: str
    price: float = Field(gt=0)           # must be a number greater than 0 (gt = 'greater than')

async def fetch_validated(pids):
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        async def one(pid):
            r = await client.get(f'/products/{pid}')
            # model_validate(dict) checks the API data against the model and returns a clean object.
            # If a field were missing or the wrong type, it would raise a clear error here.
            return ProductRecord.model_validate(r.json())
        return await asyncio.gather(*[one(pid) for pid in pids])   # fetch + validate, all at once

products = await fetch_validated([1, 2, 3])
for p in products:
    # p is now a ProductRecord object, so we read fields with a dot:  p.id, p.title, ...
    print(f'  #{p.id}  {p.title:<28} {p.category:<14} ${p.price}')


---


## 7 - Background Work: `asyncio.create_task`

`gather` waits for everything. Sometimes you want to start a job and **keep going** without waiting - like saving to the database in the background while you reply to the user. `asyncio.create_task` starts a job running on its own.


In [ ]:
log = []                                 # pretend this is our database

async def save_in_background(text):
    await asyncio.sleep(0.3)             # asyncio.sleep = a pretend wait (stands in for a real DB write)
    log.append(text)
    print(f'   (background) saved: {text!r}')

async def handle_user(message):
    # asyncio.create_task(JOB) STARTS the job running immediately, in the background,
    # and hands us back a 'task' handle WITHOUT waiting for it to finish.
    job = asyncio.create_task(save_in_background(message))
    print('Replied to the user immediately')   # we keep going right away - the save runs on its own
    # ... in a real app we'd be sending the reply here ...
    await job                            # later, 'await job' waits for the background save to finish
    print('Save confirmed.')

await handle_user('Where is my order #3042?')
print('Saved so far:', log)


> This is one of the few places we use `sleep` - and only because a throwaway 'pretend save' keeps the example short. The *real* version of this in your project saves to the database (exactly the async DB calls from section 5).


---


## 8 - Streaming an AI Reply, Word by Word

On Day 09 you saw the OpenAI API send its answer in pieces (`data: ...` lines), and we *received* them. Async is what makes streaming smooth. Here we read an OpenAI stream with `httpx.AsyncClient` and `async for`, printing each word as it arrives.

> Needs your `OPENAI_API_KEY` (from Day 09). Skips cleanly if it isn't set.


In [ ]:
OPENAI_KEY = os.environ.get('OPENAI_API_KEY')   # your key, read from the environment (Day 09)
OPENAI_URL = 'https://api.openai.com/v1/chat/completions'

async def stream_ai_reply(question):
    headers = {'Authorization': f'Bearer {OPENAI_KEY}'}   # the auth header from Day 09
    body = {
        'model': 'gpt-4o-mini',
        'messages': [{'role': 'user', 'content': question}],
        'stream': True,                  # ask OpenAI to send the answer piece by piece
    }
    #   timeout=30.0  ->  AI replies are slow, so allow up to 30 seconds (same as Day 09)
    async with httpx.AsyncClient(timeout=30.0) as client:
        # client.stream('POST', ...) keeps the connection open so we can read pieces as they arrive.
        async with client.stream('POST', OPENAI_URL, headers=headers, json=body) as resp:
            # 'async for' is the async cousin of a normal 'for' - it hands us each line the MOMENT
            # it arrives over the network, instead of waiting for the whole reply.
            async for line in resp.aiter_lines():
                if not line.startswith('data: '):
                    continue                          # skip lines that aren't data
                chunk = line[len('data: '):]          # remove the 'data: ' prefix
                if chunk == '[DONE]':                 # OpenAI sends this to mean 'finished'
                    break
                import json as _json
                # dig out the little piece of text this chunk carries (may be empty on some chunks)
                delta = _json.loads(chunk)['choices'][0]['delta']
                if 'content' in delta:
                    print(delta['content'], end='', flush=True)   # end='' so the words join up live
    print('\n(done)')

if OPENAI_KEY:
    await stream_ai_reply('List 3 quick benefits of a standing desk.')
else:
    print('(skipped - no OPENAI_API_KEY set)')


Notice `async for line in resp.aiter_lines()` - the async cousin of a normal `for`. It hands you each piece *as it arrives over the network*, so the words appear live. This is the receiving side of streaming; on the FastAPI day you'll build the *sending* side.


---


## 9 - Running Async in a Real `.py` File

Jupyter let us `await` directly. In a normal script there's no event loop yet, so you start one with `asyncio.run(...)` - one line at the bottom.


In [ ]:
reference = '''
import asyncio, httpx

async def main():
    async with httpx.AsyncClient(base_url="https://dummyjson.com") as client:
        r = await client.get("/products/1")
        print(r.json()["title"])

if __name__ == "__main__":
    asyncio.run(main())     # creates the loop, runs main(), closes it
'''
print(reference)


---


## Quick Recap - Day 10

| Tool | In one plain sentence | Built on |
|------|------------------------|----------|
| `async def` / `await` | mark 'smart waiting' work and pause politely | - |
| `httpx.AsyncClient` | the async version of Day 09's HTTP client | Day 09 |
| `asyncio.gather(*jobs)` | run many slow calls at once, answers in order | Day 09 calls |
| `return_exceptions=True` | keep good results, return errors as values | Day 09 404s |
| `asyncpg` | query Postgres the proper async way (`$1` placeholders) | Day 08 DB |
| `asyncio.to_thread(fn)` | reuse your Day 08 blocking `psycopg2` code safely | Day 08 DB |
| Pydantic `model_validate` | validate the fetched data | Day 08 |
| `asyncio.create_task(job)` | start work in the background, don't wait | - |
| `async for` + `aiter_lines` | read a stream piece by piece (AI reply) | Day 09 OpenAI |
| `asyncio.run(main())` | start async in a real `.py` file | - |

**Four things to remember:**
1. Async helps with **waiting** work (database, HTTP, AI) - exactly the slow things from Days 08 and 09.
2. `await` one line after another does **not** overlap - use `gather` to overlap.
3. For Postgres: `asyncpg` for new code, or `to_thread` to reuse your Day 08 psycopg2 code.
4. Streaming = reading pieces with `async for` as they arrive.

**Next - Day 11 (FastAPI):** you've now *called* APIs and made them fast. Next you *build* one - your own ShopSmart API that queries the database, calls the AI, and streams replies back.
